In [2]:
players = [
    {"id":1,"name":"Baran, David","rating":-7},
    {"id":2,"name":"Bi, Bowen","rating":3},
    {"id":3,"name":"Blue, Joy","rating":-22.8},
    {"id":4,"name":"Blue, Shawn","rating":-3.3},
    {"id":5,"name":"Blue, Wilton","rating":-14.2},
    {"id":6,"name":"Chen, Donovan","rating":-4.2},
    {"id":7,"name":"Chen, Leo","rating":-18},
    {"id":8,"name":"chen, shaorong","rating":1.1},
    {"id":9,"name":"Chen, Skye Eli","rating":-2},
    {"id":10,"name":"CHEN, XUDUO","rating":-1.4},
    {"id":11,"name":"chen, xuewei","rating":2.6},
    {"id":12,"name":"Cho, Christopher","rating":-13.3},
    {"id":13,"name":"Dong, David","rating":-4},
    {"id":14,"name":"Garcia, Enrique","rating":-1},
    {"id":15,"name":"Guroff, Daniel","rating":-9.2},
    {"id":16,"name":"Han, Yi","rating":-2},
    {"id":17,"name":"Hranek, Jackson T","rating":-6.3},
    {"id":18,"name":"Huang, Xingang","rating":3.7},
    {"id":19,"name":"Humbert, Christophe","rating":4.5},
    {"id":20,"name":"Kim, Christopher Christopher","rating":5.5},
    {"id":21,"name":"Kim, Youngmin","rating":-1.5},
    {"id":22,"name":"Lao, Jason Dwight C","rating":-20},
    {"id":23,"name":"Lo, Alexander","rating":-2},
    {"id":24,"name":"Lo, Benjamin","rating":-8.3},
    {"id":25,"name":"Loo, Alexander","rating":-6},
    {"id":26,"name":"Lou, James","rating":3.8},
    {"id":27,"name":"Petr, David","rating":1.1},
    {"id":28,"name":"Rich, Matt","rating":-16.6},
    {"id":29,"name":"Rudvalis, Arunas","rating":-6},
    {"id":30,"name":"Tan, Zoe","rating":-5.5},
    {"id":31,"name":"Terpstra, Theo B","rating":-6.3},
    {"id":32,"name":"Varma, Ashish","rating":3.6},
    {"id":33,"name":"Xiang, Richard","rating":-7},
    {"id":34,"name":"Yi, Harry","rating":5},
    {"id":35,"name":"Yin, Igasho","rating":4},
    {"id":36,"name":"You, Henry","rating":4},
    {"id":37,"name":"Zhang, Zeyuan","rating":4.2},
    {"id":38,"name":"Zhang, Zizhou","rating":-4.3},
    {"id":39,"name":"Zhao, Angelina","rating":2},
    {"id":40,"name":"Zhao, Bob","rating":-1},
    {"id":41,"name":"Zhou, Andy Shunwei","rating":3.5},
    {"id":42,"name":"Zhou, Angel Shunying","rating":3.5},
    {"id":43,"name":"Zhou, Anna","rating":-5.5},
    {"id":44,"name":"Zhou, Meichen","rating":-18}
]

In [37]:
import numpy as np

def distribute_numbers_with_combined_method(players, k):
    numbers = [p[1] for p in players]  # 提取rating
    n = len(numbers)
    if k > n:
        return [[num] for num in numbers]
    
    # 计算均值和标准差
    mean = np.mean(numbers)
    std = np.std(numbers)
    
    # 计算正态分布的分位数
    quantiles = np.linspace(0, 1, k + 1)[1:-1]  # 去掉 0% 和 100%
    boundaries = [np.percentile(numbers, q * 100) for q in quantiles]
    
    # 分配数字到组中
    groups = [[] for _ in range(k)]
    for num in numbers:
        for i, boundary in enumerate(boundaries):
            if num <= boundary:
                groups[i].append(num)
                break
        else:
            groups[-1].append(num)
    
    # 调整中间组为偶数
    for i in range(1, len(groups)-1):
        if len(groups[i]) % 2 != 0:
            # 尝试从前一个组移动一个数字
            if i > 0 and len(groups[i-1]) > 1:
                groups[i].insert(0, groups[i-1].pop())
            # 或者尝试从后一个组移动一个数字
            elif i < len(groups)-1 and len(groups[i+1]) > 1:
                groups[i].append(groups[i+1].pop(0))
    
    return groups

def distribute_numbers_with_normal_distribution(numbers, k):
    if k <= 0:
        return numbers
    
    n = len(numbers)
    if k > n:
        return [[num] for num in numbers]
    
    # 计算均值和标准差
    mean = np.mean(numbers)
    std = np.std(numbers)
    
    # 确定分组边界
    boundaries = []
    for i in range(1, k):
        # 使用正态分布的分位数
        quantile = i / k
        boundary = np.percentile(numbers, quantile * 100)
        boundaries.append(boundary)
    
    # 分配数字到组中
    groups = [[] for _ in range(k)]
    for num in numbers:
        for i, boundary in enumerate(boundaries):
            if num <= boundary:
                groups[i].append(num)
                break
        else:
            groups[-1].append(num)
    
    # 调整中间组为偶数
    for i in range(1, len(groups)-1):
        if len(groups[i]) % 2 != 0:
            # 将前一个组的最后一个元素移到当前组
            if i > 0 and len(groups[i-1]) > 1:
                groups[i].insert(0, groups[i-1].pop())
            # 或者将当前组的第一个元素移到后一个组
            elif i < len(groups)-1 and len(groups[i+1]) > 1:
                groups[i+1].insert(0, groups[i].pop())
    
    return groups

def auto_distribute_numbers(players, threshold=5):
    numbers = [p[1] for p in players]  # 提取rating
    n = len(numbers)
    if n < 2:
        return [numbers]
    
    # 计算相邻数字的差值
    diffs = [round(numbers[i] - numbers[i+1],1) for i in range(n-1)]
    print(diffs)
    
    # 找到差值大于阈值的分割点
    split_indices = [i for i, diff in enumerate(diffs) if diff > threshold]
    
    # 如果没有分割点，则分为 2 组
    if not split_indices:
        split_indices = [n // 2 - 1]  # 将列表分为两部分
    
    # 根据分割点分组
    groups = []
    start = 0
    for i in split_indices:
        groups.append(players[start:i+1])
        start = i+1
    groups.append(players[start:])
    
    """
    # 调整分组（除了最后一个组，其他组尽量为偶数）
    for i in range(len(groups)-1):  # 不调整最后一个组
        if len(groups[i]) % 2 != 0:
            # 尝试从前一个组移动一个数字
            if i > 0 and len(groups[i-1]) > 1:
                groups[i].insert(0, groups[i-1].pop())
            # 或者尝试从后一个组移动一个数字
            elif i < len(groups)-1 and len(groups[i+1]) > 1:
                groups[i].append(0, groups[i+1].pop(0))
    """
    
    return groups

# 示例
#numbers = [1, 1, 2, 10, 13, 15, 20, 21, 22, 23, 30, 35, 40]
numbers = [(p["name"],p["rating"]) for p in players]
numbers = sorted(numbers, key=lambda x: x[1], reverse=True)
print(numbers)
print(len(numbers))
k = 4
groups = auto_distribute_numbers(numbers, threshold=2)
for i, group in enumerate(groups):
    print(f"Group {i+1} ({len(group)}): {group}")

[('Kim, Christopher Christopher', 5.5), ('Yi, Harry', 5), ('Humbert, Christophe', 4.5), ('Zhang, Zeyuan', 4.2), ('Yin, Igasho', 4), ('You, Henry', 4), ('Lou, James', 3.8), ('Huang, Xingang', 3.7), ('Varma, Ashish', 3.6), ('Zhou, Andy Shunwei', 3.5), ('Zhou, Angel Shunying', 3.5), ('Bi, Bowen', 3), ('chen, xuewei', 2.6), ('Zhao, Angelina', 2), ('chen, shaorong', 1.1), ('Petr, David', 1.1), ('Garcia, Enrique', -1), ('Zhao, Bob', -1), ('CHEN, XUDUO', -1.4), ('Kim, Youngmin', -1.5), ('Chen, Skye Eli', -2), ('Han, Yi', -2), ('Lo, Alexander', -2), ('Blue, Shawn', -3.3), ('Dong, David', -4), ('Chen, Donovan', -4.2), ('Zhang, Zizhou', -4.3), ('Tan, Zoe', -5.5), ('Zhou, Anna', -5.5), ('Loo, Alexander', -6), ('Rudvalis, Arunas', -6), ('Hranek, Jackson T', -6.3), ('Terpstra, Theo B', -6.3), ('Baran, David', -7), ('Xiang, Richard', -7), ('Lo, Benjamin', -8.3), ('Guroff, Daniel', -9.2), ('Cho, Christopher', -13.3), ('Blue, Wilton', -14.2), ('Rich, Matt', -16.6), ('Chen, Leo', -18), ('Zhou, Meichen'